In [7]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================
# Load Dataset
# ==========================

df = pd.read_csv("mxmh_survey_results_indian_music.csv")

# ==========================
# Remove Unwanted Columns
# ==========================

drop_columns = [
    "Timestamp",
    "Music effects" ,"MusicLanguage","MusicName","Permissions" ,"Singer",
    "Frequency [Classical]",
    "Frequency [Country]",
    "Frequency [EDM]",
    "Frequency [Folk]",
    "Frequency [Gospel]",
    "Frequency [Hip hop]",
    "Frequency [Jazz]",
    "Frequency [K pop]",
    "Frequency [Latin]",
    "Frequency [Lofi]",
    "Frequency [Metal]",
    "Frequency [Pop]",
    "Frequency [R&B]",
    "Frequency [Rap]",
    "Frequency [Rock]",
    "Frequency [Video game music]"
]

df.drop(columns=drop_columns, inplace=True, errors="ignore")

# ==========================
# Handle Missing Values
# ==========================

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# ==========================
# Create Target
# ==========================

df["Listener_Type"] = (df["Hours per day"] > 4).astype(int)

# ==========================
# Prepare Features
# ==========================

X = df.drop(columns=["Hours per day", "Listener_Type"])
y = df["Listener_Type"]

# ==========================
# Encode Categorical Columns
# ==========================

encoders = {}

for col in X.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# ==========================
# Train-Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==========================
# Train Random Forest
# ==========================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

# ==========================
# Prediction
# ==========================

y_pred = model.predict(X_test)

# ==========================
# Evaluation
# ==========================

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

# ==========================
# Feature Importance
# ==========================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 Important Features\n")
print(importance.head(10))

# ==========================
# Save Model
# ==========================

joblib.dump(model, "music_listener_model.pkl")
joblib.dump(encoders, "encoders.pkl")

print("\nModel and encoders saved successfully!")

# ==========================
# Feature Names (for app.py)
# ==========================

print("\nFeatures used by the model:")
print(list(X.columns))

C:\Users\USER\AppData\Local\Temp\ipykernel_15832\1964188894.py:50: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\USER\AppData\Local\Temp\ipykernel_15832\1964188894.py:72: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

Accuracy: 0.75

Classification Report

              precision    recall  f1-score   support

           0       0.76      0.97      0.85       110
           1       0.57      0.11      0.18        38

    accuracy                           0.75       148
   macro avg       0.67      0.54      0.52       148
weighted avg       0.71      0.75      0.68       148

Confusion Matrix

[[107   3]
 [ 34   4]]

Top 10 Important Features

                      Feature  Importance
8                         BPM    0.147414
0                         Age    0.127683
10                 Depression    0.112631
5                   Fav genre    0.105581
11                   Insomnia    0.103413
9                     Anxiety    0.093174
12                        OCD    0.091645
2               While working    0.075157
1   Primary streaming service    0.050585
7           Foreign languages    0.027640

Model and encoders saved successfully!

Features used by the model:
['Age', 'Primary streaming service